In [1]:
from opt_targeted_transfers import GapTargetedTransfers
from opt_targeted_transfers import Dataset, split
from data_loaders import load_data, PATH_TO_TRAIN_DATA, PATH_TO_TEST_DATA

In [2]:
# Make train and test set
train_data = load_data(PATH_TO_TRAIN_DATA)
test_data = load_data(PATH_TO_TEST_DATA)

train_dataset = Dataset(df=train_data, outcome='consumption_per_capita_per_day', weight='hh_wgt', covs=['hh_size', 'urban'])
train_dataset, validation_dataset = split(train_dataset)
test_covariate_dataset = Dataset(df=test_data, outcome=None, weight='hh_wgt', covs=['hh_size', 'urban'])
test_dataset = Dataset(df=test_data, outcome='consumption_per_capita_per_day', weight='hh_wgt', covs=['hh_size', 'urban'])

In [3]:
# Gap targeted transfers
tt = GapTargetedTransfers(c_bar=2.15, n_regressors=5)

In [4]:
# Fit quantile regressors
tt.fit(train_dataset=train_dataset, validation_dataset=validation_dataset)

Fitting quantile regressor for quantile 0.0


100%|██████████| 300/300 [00:03<00:00, 84.73it/s, val loss=0]


Fitting quantile regressor for quantile 0.25


100%|██████████| 300/300 [00:03<00:00, 88.61it/s, val loss=0.121] 


Fitting quantile regressor for quantile 0.5


100%|██████████| 300/300 [00:03<00:00, 81.39it/s, val loss=0.194]


Fitting quantile regressor for quantile 0.75


100%|██████████| 300/300 [00:03<00:00, 79.28it/s, val loss=0.205]


Fitting quantile regressor for quantile 1.0


100%|██████████| 300/300 [00:03<00:00, 79.11it/s, val loss=0]       


In [5]:
# Get optimal policy by solving the optimization problem.
tt.set_budget(2.0)
tt.run_opt(test_covariate_dataset)
# Evaluate policy. 
res = tt.evaluate(test_dataset)
res

{'initial_poverty_rate': 0.6321457355538498,
 'initial_poverty_gap': 0.5455930541654876,
 'post_transfer_poverty_gap': 0.0,
 'post_transfer_poverty_rate': 0.0,
 'policy_cost_per_capita': 1.976034265163064,
 'budget': 2.0,
 'policy_type': 'continuous_gap',
 'd': 2}

In [6]:
tt.set_budget(0.02)
tt.run_opt(test_covariate_dataset)
# Evaluate policy. 
res = tt.evaluate(test_dataset)
res

{'initial_poverty_rate': 0.6321457355538498,
 'initial_poverty_gap': 0.5455930541654876,
 'post_transfer_poverty_gap': 0.5343519508473017,
 'post_transfer_poverty_rate': 0.6276558078604085,
 'policy_cost_per_capita': 0.01609478539109335,
 'budget': 0.02,
 'policy_type': 'continuous_gap',
 'd': 2}

In [7]:
tt.compute_auc(test_covariate_dataset=test_covariate_dataset, test_dataset=test_dataset, metrics=["post_transfer_poverty_rate",
                                                              "post_transfer_poverty_gap"], budgets=[0.05, 0.1, 0.5, 1.0, 2.0])

{'post_transfer_poverty_rate': {'auc': 0.5322825844878692,
  'results': [0.6206696923018403,
   0.6042628939579647,
   0.4538867869447938,
   0.235410182552832,
   0.0]},
 'post_transfer_poverty_gap': {'auc': 0.2844343501289715,
  'results': [0.5165406121149767,
   0.48160067281862917,
   0.24406777197614757,
   0.07110691473685211,
   0.0]}}